In [11]:
import pandas as pd

# 1. Cargar la base original
df = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital.xlsx")


In [12]:
# ---------------------------------------------------------
# 2. Identificar ganadores 2018 y 2022 por ubigeo
# ---------------------------------------------------------
ganadores = df.loc[
    df.groupby(['ubigeo', 'año'])['total_votos'].idxmax(),
    ['ubigeo', 'año', 'organizacion_politica']
]
# Nos quedamos solo con 2018 y 2022
ganadores = ganadores[ganadores['año'].isin([2018, 2022])]

# Pasar a formato ancho: una columna para ganador_2018 y otra para ganador_2022
ganadores_wide = (
    ganadores
    .pivot(index='ubigeo', columns='año', values='organizacion_politica')
    .rename(columns={2018: 'ganador_2018', 2022: 'ganador_2022'})
    .reset_index()
)


In [13]:
# ---------------------------------------------------------
# 3. Trabajar solo con datos de 2022
# ---------------------------------------------------------
df_2022 = df[df['año'] == 2022].copy()

# 3.1 Agregar los ganadores 2018 y 2022 al dataframe de 2022
df_2022 = df_2022.merge(ganadores_wide, on='ubigeo', how='left')


In [14]:
# ---------------------------------------------------------
# 4. Crear probabilidad de ganar (votos / total de votos del distrito en 2022)
# ---------------------------------------------------------
df_2022['probabilidad_ganar'] = df_2022.groupby('ubigeo')['total_votos'].transform(
    lambda x: x / x.sum()
)

# ---------------------------------------------------------
# 5. Crear turnover:
#    1 solo si:
#      - la organización de la fila es la ganadora 2022 del distrito
#      - y el ganador 2018 es el mismo que el ganador 2022
#    En todos los demás casos: 0
# ---------------------------------------------------------
df_2022['turnover'] = (
    (df_2022['organizacion_politica'] == df_2022['ganador_2022']) &
    (df_2022['ganador_2018'] == df_2022['ganador_2022'])
).astype(int)

# (Opcional) Si no quieres dejar las columnas de ganadores:
# df_2022 = df_2022.drop(columns=['ganador_2018', 'ganador_2022'])

# ---------------------------------------------------------
# 6. Guardar base final solo con 2022
# ---------------------------------------------------------
df_2022.to_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/data_partido/resultados_2022_con_probabilidad_y_turnover.xlsx", index=False)

In [15]:
# Número de casos con turnover = 1
num_turnover = df_2022['turnover'].sum()
print("Número de turnover =", num_turnover)

# (Opcional) Ver cuántos 0 y 1 hay
print(df_2022['turnover'].value_counts())

Número de turnover = 105
turnover
0    8986
1     105
Name: count, dtype: int64


In [21]:
import pandas as pd

# Cargar las dos bases
res = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/data_partido/resultados_2022_con_probabilidad_y_turnover.xlsx")
den = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/denuncias.xlsx")

# ---------------------------------------------------------
# 2. Calcular total de 'cantidad' 2018–2025 por ubigeo
# ---------------------------------------------------------
den_18_25 = (
    den[den["año"].between(2018, 2025)]  # filtra años 2018–2025
    .groupby("ubigeo")["cantidad"]       # agrupa por ubigeo
    .sum()                               # suma cantidad
    .reset_index()
    .rename(columns={"cantidad": "cantidad_2018_2025"})
)

# ---------------------------------------------------------
# 3. Unir ese total con la base de resultados por ubigeo
# ---------------------------------------------------------
res_final = res.merge(den_18_25, on="ubigeo", how="left")


# Guardar base final
res_final.to_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/data_partido/merge_con_denuncias.xlsx", index=False)


In [29]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# cargar base final
df = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/data_partido/merge_con_denuncias.xlsx")


In [30]:
# Opcional: quedarnos solo con las variables necesarias y borrar NA
df_model = df[["orden_aparicion", "turnover", "cantidad_2018_2025"]].dropna()

# 2. Ver si position (orden_aparicion) afecta turnover  →  primera etapa
X1 = sm.add_constant(df_model["orden_aparicion"])
y1 = df_model["turnover"]

modelo_pos_turnover = sm.OLS(y1, X1).fit()
print("=== Efecto de posición sobre turnover (primera etapa) ===")
print(modelo_pos_turnover.summary())

# 3. Ver si turnover afecta criminalidad (cantidad_2018_2025)  →  regresión simple
X2 = sm.add_constant(df_model["turnover"])
y2 = df_model["cantidad_2018_2025"]

modelo_turnover_crimen = sm.OLS(y2, X2).fit()
print("\n=== Efecto de turnover sobre criminalidad (regresión simple) ===")
print(modelo_turnover_crimen.summary())

# 4. Modelo IV (2SLS): posición como instrumento de turnover
#    línea causal: position → turnover → criminalidad
iv_model = IV2SLS.from_formula(
    "cantidad_2018_2025 ~ 1 + [turnover ~ orden_aparicion]",
    data=df_model
).fit()

print("\n=== Modelo IV: turnover instrumentado con posición ===")
print(iv_model.summary)

=== Efecto de posición sobre turnover (primera etapa) ===
                            OLS Regression Results                            
Dep. Variable:               turnover   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     10.17
Date:                Sun, 30 Nov 2025   Prob (F-statistic):            0.00144
Time:                        15:28:51   Log-Likelihood:                 4615.8
No. Observations:                5010   AIC:                            -9228.
Df Residuals:                    5008   BIC:                            -9214.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------